In [0]:
%pip install hotel_reservation-0.0.1-py3-none-any.whl

Processing ./hotel_reservation-0.0.1-py3-none-any.whl
INFO: pip is looking at multiple versions of mlflow-skinny[databricks] to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of mlflow-skinny[databricks] to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of databricks-agents to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of google-api-core to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of grpcio-status to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 681.8/681.8 kB 34.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 136.5 MB/s eta 0:00:00
 

In [0]:
%restart_python

In [0]:
import hashlib
import os
import time

import mlflow
import requests
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import (
    EndpointCoreConfigInput,
    ServedEntityInput,
)
from dotenv import load_dotenv
from mlflow.models import infer_signature
from pyspark.sql import SparkSession

from hotel_reservation.config import ProjectConfig, Tags
from hotel_reservation.models.basic_model import BasicModel
from hotel_reservation.utils import is_databricks

In [0]:
if not is_databricks():
    load_dotenv()
    profile = os.environ.get("PROFILE", "DEFAULT")
    mlflow.set_tracking_uri(f"databricks://{profile}")
    mlflow.set_registry_uri(f"databricks-uc://{profile}")


config = ProjectConfig.from_yaml(config_path="../project_config.yml", env="prd")
spark = SparkSession.builder.getOrCreate()
tags = Tags(**{"git_sha": "abcd12345", "branch": "week4"})

In [0]:
# Load project config
config = ProjectConfig.from_yaml(config_path="../project_config.yml", env="dev")
catalog_name = config.catalog_name
schema_name = config.schema_name

In [0]:
# train model A
basic_model = BasicModel(config=config, tags=tags, spark=spark)
basic_model.load_data()
basic_model.prepare_features()
basic_model.train()
basic_model.log_model()
basic_model.register_model()
model_A_uri = f"models:/{basic_model.model_name}@latest-model"

2025-09-30 14:22:28.714 | INFO     | hotel_reservation.models.basic_model:load_data:61 - 🔄 Loading data from Databricks tables...
2025-09-30 14:22:30.336 | INFO     | hotel_reservation.models.basic_model:load_data:71 - ✅ Data successfully loaded.
2025-09-30 14:22:30.347 | INFO     | hotel_reservation.models.basic_model:load_data:76 - ✅ Target successfully encoded.
2025-09-30 14:22:30.348 | INFO     | hotel_reservation.models.basic_model:prepare_features:84 - 🔄 Defining preprocessing pipeline...
2025-09-30 14:22:30.350 | INFO     | hotel_reservation.models.basic_model:prepare_features:92 - ✅ Preprocessing pipeline defined.
2025-09-30 14:22:30.351 | INFO     | hotel_reservation.models.basic_model:train:96 - 🚀 Starting training...


[LightGBM] [Info] Number of positive: 19551, number of negative: 9469
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.004929 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 660
[LightGBM] [Info] Number of data points in the train set: 29020, number of used features: 28
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.673708 -> initscore=0.725003
[LightGBM] [Info] Start training from score 0.725003
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -

/local_disk0/.ephemeral_nfs/envs/pythonEnv-c6a7bba6-1135-4b63-be45-4f3979067649/lib/python3.12/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
2025-09-30 14:22:34.053 | INFO     | hotel_reservation.models.basic_model:log_model:112 - 📊 Accuracy: 0.8745692625775328
2025-09-30 14:22:34.054 | INFO     | hotel_reservation.models.basic_model:log_model:113 - 📊 Precision: 0.8872462054011433
2025-09-30 14:22:34.054 | INFO     | hotel_reservation.models.basic_model:log_model:114 - 📊 Recall: 0.9301508576152098
/local_disk0/.ephemeral_nfs/envs/pythonEnv-c6a7bba6-1135-4b63-be45-4f3979067649/lib/python3.12/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforc

Uploading artifacts:   0%|          | 0/10 [00:00<?, ?it/s]

🔗 Created version '2' of model 'mlops_dev.nikhilko.hotel_reservation_model_basic': https://dbc-f122dc18-1b68.cloud.databricks.com/explore/data/models/mlops_dev/nikhilko/hotel_reservation_model_basic/version/2?o=2661948581729539
2025-09-30 14:22:47.244 | INFO     | hotel_reservation.models.basic_model:register_model:141 - ✅ Model registered as version 2.


In [0]:
# train model B
basic_model_b = BasicModel(config=config, tags=tags, spark=spark)
basic_model_b.paramaters = {"learning_rate": 0.1,
                            "n_estimators": 1200,
                            "max_depth": 5}
basic_model_b.model_name = f"{catalog_name}.{schema_name}.hotel_reservation_model_basic_B"
basic_model_b.load_data()
basic_model_b.prepare_features()
basic_model_b.train()
basic_model_b.log_model()
basic_model_b.register_model()
model_B_uri = f"models:/{basic_model_b.model_name}@latest-model"

2025-09-30 14:26:16.967 | INFO     | hotel_reservation.models.basic_model:load_data:61 - 🔄 Loading data from Databricks tables...
2025-09-30 14:26:17.778 | INFO     | hotel_reservation.models.basic_model:load_data:71 - ✅ Data successfully loaded.
2025-09-30 14:26:17.784 | INFO     | hotel_reservation.models.basic_model:load_data:76 - ✅ Target successfully encoded.
2025-09-30 14:26:17.784 | INFO     | hotel_reservation.models.basic_model:prepare_features:84 - 🔄 Defining preprocessing pipeline...
2025-09-30 14:26:17.785 | INFO     | hotel_reservation.models.basic_model:prepare_features:92 - ✅ Preprocessing pipeline defined.
2025-09-30 14:26:17.786 | INFO     | hotel_reservation.models.basic_model:train:96 - 🚀 Starting training...


[LightGBM] [Info] Number of positive: 19551, number of negative: 9469
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.003106 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 660
[LightGBM] [Info] Number of data points in the train set: 29020, number of used features: 28
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.673708 -> initscore=0.725003
[LightGBM] [Info] Start training from score 0.725003
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -

/local_disk0/.ephemeral_nfs/envs/pythonEnv-c6a7bba6-1135-4b63-be45-4f3979067649/lib/python3.12/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
2025-09-30 14:26:20.664 | INFO     | hotel_reservation.models.basic_model:log_model:112 - 📊 Accuracy: 0.8745692625775328
2025-09-30 14:26:20.667 | INFO     | hotel_reservation.models.basic_model:log_model:113 - 📊 Precision: 0.8872462054011433
2025-09-30 14:26:20.669 | INFO     | hotel_reservation.models.basic_model:log_model:114 - 📊 Recall: 0.9301508576152098
/local_disk0/.ephemeral_nfs/envs/pythonEnv-c6a7bba6-1135-4b63-be45-4f3979067649/lib/python3.12/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforc

Uploading artifacts:   0%|          | 0/10 [00:00<?, ?it/s]

🔗 Created version '1' of model 'mlops_dev.nikhilko.hotel_reservation_model_basic_b': https://dbc-f122dc18-1b68.cloud.databricks.com/explore/data/models/mlops_dev/nikhilko/hotel_reservation_model_basic_b/version/1?o=2661948581729539
2025-09-30 14:26:29.955 | INFO     | hotel_reservation.models.basic_model:register_model:141 - ✅ Model registered as version 1.


In [0]:
# define wrapper
class HotelReservationModelWrapper(mlflow.pyfunc.PythonModel):
    def load_context(self, context):
        self.model_a = mlflow.sklearn.load_model(
            context.artifacts["lightgbm-pipeline-model-A"]
        )
        self.model_b = mlflow.sklearn.load_model(
            context.artifacts["lightgbm-pipeline-model-B"]
        )

    def predict(self, context, model_input):
        booking_id = str(model_input["Booking_ID"].values[0])
        hashed_id = hashlib.md5(booking_id.encode(encoding="UTF-8")).hexdigest()
        # convert a hexadecimal (base-16) string into an integer
        if int(hashed_id, 16) % 2:
            predictions = self.model_a.predict(model_input.drop(["Booking_ID"], axis=1))
            return {"Prediction": predictions[0], "model": "Model A"}
        else:
            predictions = self.model_b.predict(model_input.drop(["Booking_ID"], axis=1))
            return {"Prediction": predictions[0], "model": "Model B"}

/local_disk0/.ephemeral_nfs/envs/pythonEnv-c6a7bba6-1135-4b63-be45-4f3979067649/lib/python3.12/site-packages/mlflow/pyfunc/utils/data_validation.py:186: UserWarning: Add type hints to the `predict` method to enable data validation and automatic signature inference during model logging. Check https://mlflow.org/docs/latest/model/python_model.html#type-hint-usage-in-pythonmodel for more details.
  color_warning(


In [0]:
train_set_spark = spark.table(f"{catalog_name}.{schema_name}.train_set")
train_set = train_set_spark.toPandas()
test_set = spark.table(f"{catalog_name}.{schema_name}.test_set").toPandas()
X_train = train_set[config.num_features + config.cat_features + ["Booking_ID"]]
X_test = test_set[config.num_features + config.cat_features + ["Booking_ID"]]

In [0]:
mlflow.set_experiment(experiment_name="/Shared/hotel-reservations-ab-testing")
model_name = f"{catalog_name}.{schema_name}.hotel_reservations_model_pyfunc_ab_test"
wrapped_model = HotelReservationModelWrapper()

with mlflow.start_run() as run:
    run_id = run.info.run_id
    signature = infer_signature(model_input=X_train, model_output={"Prediction": [0], "model": "Model B"})
    dataset = mlflow.data.from_spark(train_set_spark, table_name=f"{catalog_name}.{schema_name}.train_set", version="0")
    mlflow.log_input(dataset, context="training")
    mlflow.pyfunc.log_model(
        python_model=wrapped_model,
        artifact_path="pyfunc-hotel-reservation-model-ab",
        artifacts={
            "lightgbm-pipeline-model-A": model_A_uri,
            "lightgbm-pipeline-model-B": model_B_uri},
        signature=signature
    )
model_version = mlflow.register_model(
    model_uri=f"runs:/{run_id}/pyfunc-hotel-reservation-model-ab", name=model_name, tags=tags.dict()
)

2025/09/30 14:30:23 INFO mlflow.tracking.fluent: Experiment with name '/Shared/hotel-reservations-ab-testing' does not exist. Creating a new experiment.
/local_disk0/.ephemeral_nfs/envs/pythonEnv-c6a7bba6-1135-4b63-be45-4f3979067649/lib/python3.12/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2025/09/30 14:30:24 WARNIN

/local_disk0/.ephemeral_nfs/envs/pythonEnv-c6a7bba6-1135-4b63-be45-4f3979067649/lib/python3.12/site-packages/mlflow/pyfunc/__init__.py:3262: UserWarning: An input example was not provided when logging the model. To ensure the model signature functions correctly, specify the `input_example` parameter. See https://mlflow.org/docs/latest/model/signatures.html#model-input-example for more details about the benefits of using input_example.
  color_warning(


/home/spark-c6a7bba6-1135-4b63-be45-4f/.ipykernel/6624/command-6316357214901036-2326607766:19: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.11/migration/
  model_uri=f"runs:/{run_id}/pyfunc-hotel-reservation-model-ab", name=model_name, tags=tags.dict()
Successfully registered model 'mlops_dev.nikhilko.hotel_reservations_model_pyfunc_ab_test'.
2025/09/30 14:30:37 WARNING mlflow.tracking._model_registry.fluent: Run with id c9f34f77754f4d2f9509ac2dc3de29bc has no artifacts at artifact path 'pyfunc-hotel-reservation-model-ab', registering model based on models:/m-0ef4f5c0cd324a0291c3bdfd6ed93b03 instead


Uploading artifacts:   0%|          | 0/20 [00:00<?, ?it/s]

🔗 Created version '1' of model 'mlops_dev.nikhilko.hotel_reservations_model_pyfunc_ab_test': https://dbc-f122dc18-1b68.cloud.databricks.com/explore/data/models/mlops_dev/nikhilko/hotel_reservations_model_pyfunc_ab_test/version/1?o=2661948581729539


In [0]:
"""Model serving module."""

workspace = WorkspaceClient()
model_name=f"{catalog_name}.{schema_name}.hotel_reservations_model_pyfunc_ab_test"
endpoint_name="hotel-reservations-ab-testing"
entity_version = model_version.version # registered model version

# get environment variables
os.environ["DBR_HOST"] = workspace.config.host
os.environ["DBR_TOKEN"] = workspace.tokens.create(lifetime_seconds=1200).token_value

served_entities = [
    ServedEntityInput(
        entity_name=model_name,
        scale_to_zero_enabled=True,
        workload_size="Small",
        entity_version=entity_version,
    )
]

workspace.serving_endpoints.create(
        name=endpoint_name,
        config=EndpointCoreConfigInput(
            served_entities=served_entities,
        ),
    )

In [0]:
# Create a sample request body

spark = SparkSession.builder.getOrCreate()

train_set = spark.table(f"{catalog_name}.{schema_name}.train_set").toPandas()
sampled_records = train_set[config.num_features + config.cat_features + ["Booking_ID"]].sample(n=1000, replace=True).to_dict(orient="records")
dataframe_records = [[record] for record in sampled_records]

print(train_set.dtypes)
print(dataframe_records[0])

type_of_meal_plan                               object
room_type_reserved                              object
market_segment_type                             object
no_of_adults                                     int64
no_of_children                                   int64
no_of_weekend_nights                             int64
no_of_week_nights                                int64
required_car_parking_space                       int64
lead_time                                        int64
arrival_year                                     int64
arrival_month                                    int64
arrival_date                                     int64
repeated_guest                                   int64
no_of_previous_cancellations                     int64
no_of_previous_bookings_not_canceled             int64
avg_price_per_room                             float64
no_of_special_requests                           int64
booking_status                                  object
Booking_ID

In [0]:
# Call the endpoint with one sample record

def call_endpoint(record):
    """Calls the model serving endpoint with a given input record."""
    serving_endpoint = f"{os.environ['DBR_HOST']}/serving-endpoints/{endpoint_name}/invocations"

    response = requests.post(
        serving_endpoint,
        headers={"Authorization": f"Bearer {os.environ['DBR_TOKEN']}"},
        json={"dataframe_records": record},
    )
    return response.status_code, response.text


status_code, response_text = call_endpoint(dataframe_records[0])
print(f"Response Status: {status_code}")
print(f"Response Text: {response_text}")

Response Status: 200
Response Text: {"predictions": {"Prediction": 1, "model": "Model A"}}


In [0]:
# Load test
for i in range(len(dataframe_records)):
    status_code, response_text = call_endpoint(dataframe_records[i])
    print(f"Response Status: {status_code}")
    print(f"Response Text: {response_text}")
    time.sleep(0.2)

Response Status: 200
Response Text: {"predictions": {"Prediction": 1, "model": "Model A"}}
Response Status: 200
Response Text: {"predictions": {"Prediction": 1, "model": "Model A"}}
Response Status: 200
Response Text: {"predictions": {"Prediction": 1, "model": "Model B"}}
Response Status: 200
Response Text: {"predictions": {"Prediction": 0, "model": "Model A"}}
Response Status: 200
Response Text: {"predictions": {"Prediction": 0, "model": "Model A"}}
Response Status: 200
Response Text: {"predictions": {"Prediction": 0, "model": "Model B"}}
Response Status: 200
Response Text: {"predictions": {"Prediction": 0, "model": "Model B"}}
Response Status: 200
Response Text: {"predictions": {"Prediction": 0, "model": "Model A"}}
Response Status: 200
Response Text: {"predictions": {"Prediction": 0, "model": "Model A"}}
Response Status: 200
Response Text: {"predictions": {"Prediction": 0, "model": "Model B"}}
Response Status: 200
Response Text: {"predictions": {"Prediction": 1, "model": "Model B"}}